# Gated Consensus Ensemble — Option A: Per-Cluster Method Overrides

Extends the base gated ensemble with a **manual override dict** that lets
the user force which methods are allowed on a per-cluster basis. This is
useful when visual inspection reveals that a particular method produces
nonsense for a cluster even though its scores technically pass the
threshold — or conversely, that only one method actually tracks the
pattern.

**Two layers of gating:**
1. **Score thresholds** (same as base notebook) — a gene must pass the
   method's quality bar.
2. **Cluster overrides** — the user can restrict which methods are even
   *allowed* to vote for a given cluster. If a cluster appears in the
   override dict, only the listed methods participate. If the list is
   empty, the cluster is flagged as "no consensus possible."

Only LT (log-transformed) data is used.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
import pickle

%matplotlib inline

results = pickle.load(open('metrics_results.pkl', 'rb'))
constraint_patternsLT = results['constraint_patternsLT']
all_methodsLT = results['all_methodsLT']
method_names = list(all_methodsLT.keys())

print('Methods:', method_names)
print('Clusters:', list(constraint_patternsLT.keys()))

## 2. Thresholds (editable)

Each method gets a **(direction, value)** pair:
- `('>', 0.99)` means the gene's score must be **above** 0.99 to qualify
- `('<', 0.08)` means the gene's score must be **below** 0.08 to qualify

Tune these values and re-run cells below.

In [ ]:
THRESHOLDS = {
    'Pearson':  ('>', 0.99),
    'DTW':      ('<', 0.08),
    'Cosine':   ('>', 0.999),
    'Frechet':  ('<', 0.03),
    'MSE':      ('<', 0.001),
    'sMAPE':    ('<', 0.10),
}

TOP_N = 20

print('Thresholds:')
for m, (op, val) in THRESHOLDS.items():
    print(f'  {m:>8}: score {op} {val}')

## 3. Cluster overrides (editable)

Keys are cluster names, values are lists of **allowed** method names.

- If a cluster is **not** in this dict, all 6 methods are allowed
  (subject to thresholds).
- If a cluster maps to an **empty list**, it is flagged as
  "no consensus possible" and skipped.
- If a cluster maps to a non-empty list, only those methods may vote.

Edit this dict based on your visual inspection of per-method plots.

In [ ]:
CLUSTER_OVERRIDES = {
    'clusterThreeLT': ['sMAPE'],           # only sMAPE overlays the pattern
    'clusterFourLT':  [],                   # no method works -- flag as no consensus
}

print('Cluster overrides:')
for cluster, allowed in CLUSTER_OVERRIDES.items():
    if len(allowed) == 0:
        print(f'  {cluster}: NO METHODS ALLOWED (flagged as no consensus)')
    else:
        print(f'  {cluster}: {allowed}')
print()
unconstrained = [c for c in constraint_patternsLT if c not in CLUSTER_OVERRIDES]
if unconstrained:
    print(f'Clusters with all methods allowed: {unconstrained}')

## 4. Qualifying genes (thresholds + overrides)

In [ ]:
def get_qualifying_genes(methods_dict, cluster_name, pattern_name,
                         thresholds, overrides=None):
    """For each method, return the set of genes that pass its threshold,
    respecting per-cluster overrides.

    Parameters
    ----------
    methods_dict : dict
        method_name -> (scores_dict, ascending_bool)
    cluster_name : str
    pattern_name : str
    thresholds : dict
        method_name -> (operator_str, threshold_value)
    overrides : dict or None
        cluster_name -> list of allowed method names.
        If cluster not in overrides, all methods are allowed.
        If cluster maps to [], no methods are allowed.

    Returns
    -------
    qualifying : dict
        method_name -> Series of scores (only qualifying genes), sorted.
    silenced_by_override : list
        Methods silenced because of the override (not threshold).
    """
    if overrides is None:
        overrides = {}

    # Determine which methods are allowed for this cluster
    if cluster_name in overrides:
        allowed = set(overrides[cluster_name])
    else:
        allowed = set(methods_dict.keys())

    silenced_by_override = [m for m in methods_dict if m not in allowed]

    qualifying = {}
    for mname, (scores, ascending) in methods_dict.items():
        if mname not in thresholds:
            continue
        if mname not in allowed:
            continue  # silenced by override
        op, val = thresholds[mname]
        s = scores[cluster_name][pattern_name]
        if op == '>':
            mask = s > val
        elif op == '<':
            mask = s < val
        elif op == '>=':
            mask = s >= val
        elif op == '<=':
            mask = s <= val
        else:
            raise ValueError(f'Unknown operator: {op}')
        qualifying[mname] = s[mask].sort_values(ascending=ascending)
    return qualifying, silenced_by_override


# Show qualifying gene counts per method x cluster
print(f'{"=" * 70}')
print(f'LT DATA -- qualifying gene counts (with overrides)')
print(f'{"=" * 70}')
for cluster_name in constraint_patternsLT:
    q, silenced_override = get_qualifying_genes(
        all_methodsLT, cluster_name, 'constraint',
        THRESHOLDS, CLUSTER_OVERRIDES
    )
    override_tag = ''
    if cluster_name in CLUSTER_OVERRIDES:
        allowed = CLUSTER_OVERRIDES[cluster_name]
        if len(allowed) == 0:
            override_tag = '  [OVERRIDE: no methods allowed]'
        else:
            override_tag = f'  [OVERRIDE: only {allowed}]'
    print(f'\n  {cluster_name}:{override_tag}')
    for mname in method_names:
        if mname in silenced_override:
            print(f'    {mname:>8}: SILENCED BY OVERRIDE')
        elif mname in q:
            n = len(q[mname])
            op, val = THRESHOLDS[mname]
            tag = ' ** SILENCED BY THRESHOLD **' if n == 0 else ''
            print(f'    {mname:>8} ({op}{val}): {n:>6,} genes{tag}')
        else:
            print(f'    {mname:>8}: not configured')

## 5. Plot qualifying genes per method (active methods only)

In [ ]:
def plot_qualifying_genes_optA(methods_dict, pat_dict, thresholds,
                               overrides=None, max_plot=50):
    """For each cluster x active method, plot qualifying genes overlaid on pattern."""
    x = [0, 3, 6, 9]

    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_countsLT/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x

        for pattern_name, pattern_vals in pattern_dict.items():
            q, silenced_override = get_qualifying_genes(
                methods_dict, cluster_name, pattern_name,
                thresholds, overrides
            )
            active = [m for m in method_names if m in q and len(q[m]) > 0]
            silenced_thresh = [m for m in method_names
                               if m in q and len(q[m]) == 0]
            n_active = len(active)

            if n_active == 0:
                reason = 'ALL METHODS SILENCED'
                if cluster_name in (overrides or {}):
                    allowed = (overrides or {})[cluster_name]
                    if len(allowed) == 0:
                        reason += ' (override: no methods allowed)'
                    else:
                        reason += ' (override + threshold)'
                print(f'{cluster_name}: {reason}')
                continue

            fig, axes = plt.subplots(1, n_active,
                                     figsize=(5 * n_active, 4.5),
                                     squeeze=False)

            for i, mname in enumerate(active):
                ax = axes[0, i]
                genes = q[mname].index.tolist()
                n_genes = len(genes)
                plotted = 0
                for gene in genes[:max_plot]:
                    if gene in gene_df.index:
                        ax.plot(x, gene_df.loc[gene], color='steelblue',
                                alpha=0.3, linewidth=0.8)
                        plotted += 1
                ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                        linestyle='--')
                op, val = thresholds[mname]
                ax.set_title(f'{mname} ({op}{val})\n{n_genes:,} genes qualify',
                             fontsize=10)
                ax.set_xlabel('Timepoint')
                ax.set_xticks(x)
                if i == 0:
                    ax.set_ylabel('Gene counts (LT)')
                ax.grid(alpha=0.2)

            all_silenced = sorted(set(silenced_override) | set(silenced_thresh))
            sil_parts = []
            for m in all_silenced:
                if m in silenced_override:
                    sil_parts.append(f'{m} (override)')
                else:
                    sil_parts.append(f'{m} (threshold)')
            sil_str = ', '.join(sil_parts) if sil_parts else 'none'
            fig.suptitle(
                f'{cluster_name} (LT) -- Option A\nSilenced: {sil_str}',
                fontsize=12, fontweight='bold'
            )
            plt.tight_layout()
            plt.show()


plot_qualifying_genes_optA(all_methodsLT, constraint_patternsLT,
                           THRESHOLDS, CLUSTER_OVERRIDES)

## 6. Gated consensus with overrides

A gene gets a vote from a method **only if**:
1. The method is **allowed** for this cluster (not blocked by override), AND
2. The gene's score **passes** that method's threshold.

In [ ]:
def gated_consensus_optA(methods_dict, cluster_name, pattern_name,
                         thresholds, overrides=None, top_n=20):
    """Score-gated ensemble with per-cluster method overrides.

    Returns
    -------
    result : DataFrame
        Top-N genes with votes, max_possible, mean_rank columns.
    active : list
        Methods that are allowed AND have >= 1 qualifying gene.
    silenced_override : list
        Methods blocked by override dict.
    silenced_thresh : list
        Methods allowed by override but with 0 qualifying genes.
    """
    q, silenced_override = get_qualifying_genes(
        methods_dict, cluster_name, pattern_name,
        thresholds, overrides
    )
    active = [m for m in method_names if m in q and len(q[m]) > 0]
    silenced_thresh = [m for m in method_names
                       if m in q and len(q[m]) == 0]

    if len(active) == 0:
        return pd.DataFrame(), active, silenced_override, silenced_thresh

    # Build rank df for active methods only
    rank_df = pd.DataFrame()
    for mname in active:
        scores, ascending = methods_dict[mname]
        s = scores[cluster_name][pattern_name]
        rank_df[mname] = s.rank(ascending=ascending, method='min')

    # A gene gets a vote from a method only if it's in that method's qualifying set
    vote_df = pd.DataFrame(index=rank_df.index)
    for mname in active:
        qualifying_genes = set(q[mname].index)
        vote_df[mname] = rank_df.index.isin(qualifying_genes).astype(int)

    votes = vote_df.sum(axis=1)
    mean_rank = rank_df[active].mean(axis=1)

    result = pd.DataFrame({
        'votes': votes,
        'max_possible': len(active),
        'mean_rank': mean_rank,
    })
    # Only keep genes with at least 1 vote
    result = result[result['votes'] > 0]
    result = result.sort_values(['votes', 'mean_rank'],
                                ascending=[False, True])

    return result.head(top_n), active, silenced_override, silenced_thresh

## 7. Results table

In [ ]:
print('GATED CONSENSUS ENSEMBLE -- OPTION A (LT DATA)')
print('=' * 90)

for cluster_name in constraint_patternsLT:
    result, active, sil_over, sil_thresh = gated_consensus_optA(
        all_methodsLT, cluster_name, 'constraint',
        THRESHOLDS, CLUSTER_OVERRIDES, top_n=TOP_N
    )

    print(f'\n{"=" * 90}')
    print(f'{cluster_name}')
    print(f'{"=" * 90}')

    # Override info
    if cluster_name in CLUSTER_OVERRIDES:
        allowed = CLUSTER_OVERRIDES[cluster_name]
        if len(allowed) == 0:
            print(f'  Override: NO METHODS ALLOWED')
        else:
            print(f'  Override: only {allowed}')
    else:
        print(f'  Override: none (all methods allowed)')

    print(f'  Active methods:   {len(active)}/{len(method_names)} -- {", ".join(active) if active else "NONE"}')

    if sil_over:
        print(f'  Silenced (override):  {", ".join(sil_over)}')
    if sil_thresh:
        print(f'  Silenced (threshold): {", ".join(sil_thresh)}')

    if result.empty:
        print(f'\n  ** NO QUALIFYING GENES -- no consensus possible **')
        continue

    # Get qualifying info for detailed reporting
    q, _ = get_qualifying_genes(
        all_methodsLT, cluster_name, 'constraint',
        THRESHOLDS, CLUSTER_OVERRIDES
    )

    max_v = len(active)
    print(f'\n  {"Rank":<6}{"Gene":<22}{"Votes":>7}{"Mean Rank":>11}  Qualified in')
    print(f'  {"-" * 80}')
    for i, (gene, row) in enumerate(result.iterrows(), 1):
        v = int(row['votes'])
        mr = row['mean_rank']
        in_methods = [m for m in active if gene in q.get(m, pd.Series()).index]
        print(f'  {i:<6}{gene:<22}{v:>5}/{max_v}  {mr:>9.1f}  {", ".join(in_methods)}')

## 8. Ensemble plots (opacity by vote count)

In [ ]:
def plot_gated_consensus_optA(methods_dict, pat_dict, thresholds,
                              overrides=None, top_n=20):
    """Plot gated consensus top-N genes. Opacity = vote fraction."""
    x = [0, 3, 6, 9]

    for cluster_name, pattern_dict in pat_dict.items():
        gene_file = f'analysis data/gene_countsLT/{cluster_name}_annotated.csv'
        gene_df = pd.read_csv(gene_file, index_col=0)
        gene_df.columns = x

        for pattern_name, pattern_vals in pattern_dict.items():
            result, active, sil_over, sil_thresh = gated_consensus_optA(
                methods_dict, cluster_name, pattern_name,
                thresholds, overrides, top_n=top_n
            )
            if result.empty:
                print(f'{cluster_name}: no consensus -- skipping plot')
                continue

            max_votes = len(active)
            fig, ax = plt.subplots(figsize=(10, 6))

            for gene in result.index:
                if gene in gene_df.index:
                    v = result.loc[gene, 'votes']
                    alpha = 0.2 + 0.6 * (v / max_votes)
                    ax.plot(x, gene_df.loc[gene], color='steelblue',
                            alpha=alpha, linewidth=1)

            ax.plot(x, pattern_vals, color='crimson', linewidth=2.5,
                    linestyle='--')

            # Build title with silencing info
            all_sil = []
            for m in sil_over:
                all_sil.append(f'{m}(ovr)')
            for m in sil_thresh:
                all_sil.append(f'{m}(thr)')
            sil_str = ', '.join(all_sil) if all_sil else 'none'

            title = (
                f'Gated Consensus (Option A) | {cluster_name} (LT) | '
                f'Top {top_n} genes\n'
                f'Active: {", ".join(active)} | '
                f'Silenced: {sil_str}'
            )
            ax.set_title(title, fontsize=10)
            ax.set_xlabel('Timepoint')
            ax.set_ylabel('Gene counts (LT)')
            ax.set_xticks(x)
            ax.grid(alpha=0.2)

            legend_elements = [
                Line2D([0], [0], color='crimson', linewidth=2.5,
                       linestyle='--', label='Constraint pattern'),
                Line2D([0], [0], color='steelblue', alpha=0.8,
                       label=f'High consensus (>={max_votes - 1}/{max_votes} votes)'),
                Line2D([0], [0], color='steelblue', alpha=0.3,
                       label='Low consensus (1 vote)'),
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            plt.tight_layout()
            plt.show()


plot_gated_consensus_optA(all_methodsLT, constraint_patternsLT,
                          THRESHOLDS, CLUSTER_OVERRIDES, top_n=TOP_N)